# Regression with PyTorch v4 (oracle) — Income, Neural Additive Model + robust head

**Version:** 4th Version (novel architectures). **Role:** reference oracle for the v4 scratch twin.
Same split/seed as v2, full 13-dim set. Model: 13 per-feature subnets `Linear(1->64)->ReLU->Linear(64->1)`, sum + bias (glass-box: each feature's shape function plottable). Target: `log1p(Income)` z-scored (train-fit scaler), Huber(delta=0.5) in z-log space, `expm1` inversion for $ metrics. The $666,666 outlier is KEPT — robustness comes from log+Huber math, not deletion (masked diagnostic reported separately). Adam + cosine + wd, 100 epochs, best-val-RMSE checkpoint. Weights → `./weights_v4/`.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_v4", exist_ok=True)
DELTA = 0.5

df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

X_temp, X_test, y_temp_raw, y_test_raw = train_test_split(X_full, y_raw, test_size=0.20, random_state=SEED)
scaler = StandardScaler().fit(X_temp)
Xtr, Xte = scaler.transform(X_temp), scaler.transform(X_test)
y_temp_log = np.log1p(y_temp_raw).reshape(-1, 1); y_test_log = np.log1p(y_test_raw).reshape(-1, 1)
y_scaler = StandardScaler().fit(y_temp_log)
ytr_z = y_scaler.transform(y_temp_log).ravel(); yte_z = y_scaler.transform(y_test_log).ravel()
print(f"train {Xtr.shape} test {Xte.shape} | y_log scaler: mean {y_scaler.mean_[0]:.4f} scale {y_scaler.scale_[0]:.4f}")
train_ds, test_ds = TabularDataset(Xtr, ytr_z), TabularDataset(Xte, yte_z)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

class SubNet(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.fc1 = nn.Linear(1, h); self.fc2 = nn.Linear(h, 1); self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu"); nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu"); nn.init.zeros_(self.fc2.bias)
    def forward(self, xj): return self.fc2(self.relu(self.fc1(xj)))

class NAM(nn.Module):
    def __init__(self, n_feat=13, h=64):
        super().__init__()
        self.subs = nn.ModuleList([SubNet(h) for _ in range(n_feat)])
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return sum(sub(x[:, j:j+1]) for j, sub in enumerate(self.subs)) + self.b0

device = torch.device("cpu")
model = NAM(n_feat=len(REG_FEATURES), h=64).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"NAM params: {n_params}")
criterion = nn.HuberLoss(delta=DELTA)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
print(f"Huber delta={DELTA} | epochs 100 | cosine | wd 1e-4 | log1p target, outlier kept")

def to_dollar(z):
    return np.expm1(y_scaler.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel())
def evaluate(m):
    m.eval()
    with torch.no_grad():
        Xt = torch.from_numpy(Xte.astype(np.float32)).to(device)
        pred_z = m(Xt).cpu().numpy().ravel()
    pred = to_dollar(pred_z)
    rmse = float(np.sqrt(mean_squared_error(y_test_raw, pred)))
    return rmse, float(mean_absolute_error(y_test_raw, pred)), float(r2_score(y_test_raw, pred))

EPOCHS = 100
history = {"epoch": [], "train_loss": [], "val_huber": [], "val_rmse": [], "val_r2": [], "lr": []}
best_rmse, best_state = float("inf"), None
for epoch in range(1, EPOCHS + 1):
    model.train(); tot, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
        tot += loss.item() * len(xb); n += len(xb)
    scheduler.step()
    rmse, mae, r2 = evaluate(model)
    lr = optimizer.param_groups[0]["lr"]
    history["epoch"].append(epoch); history["train_loss"].append(tot / n)
    history["val_huber"].append(float(tot / n)); history["val_rmse"].append(rmse); history["val_r2"].append(r2); history["lr"].append(lr)
    if rmse < best_rmse: best_rmse = rmse; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:03d} | train-huber {tot/n:.4f} | val-RMSE ${rmse:,.0f} | R2 {r2:.4f} | lr {lr:.2e} | bestRMSE ${best_rmse:,.0f}")
torch.save(best_state, "weights_v4/regression_nam.pt")
print(f"best val-RMSE checkpoint saved (${best_rmse:,.0f})")
np.savez("weights_v4/regression_scaler.npz", mean=scaler.mean_, scale=scaler.scale_,
         columns=np.array(REG_FEATURES), y_mean=y_scaler.mean_, y_scale=y_scaler.scale_)
pd.DataFrame(history).to_csv("weights_v4/regression_pytorch_history.csv", index=False)

fresh = NAM(n_feat=len(REG_FEATURES), h=64).to(device)
fresh.load_state_dict(torch.load("weights_v4/regression_nam.pt", map_location=device))
fresh.eval()
with torch.no_grad():
    pred = to_dollar(fresh(torch.from_numpy(Xte.astype(np.float32)).to(device)).cpu().numpy().ravel())
mse = float(mean_squared_error(y_test_raw, pred))
metrics = {"test_MSE": round(mse, 1), "RMSE": round(float(np.sqrt(mse)), 1),
    "MAE": round(float(mean_absolute_error(y_test_raw, pred)), 1),
    "R2": round(float(r2_score(y_test_raw, pred)), 4),
    "n_test": int(len(y_test_raw)), "seed": SEED, "delta": DELTA, "target": "log1p+Huber"}
print(json.dumps(metrics, indent=2))
with open("weights_v4/regression_pytorch_test_metrics.json", "w") as f: json.dump(metrics, f, indent=2)

# outlier diagnostic: metrics with top-9 >$118k rows masked (report-only, model unchanged)
mask = y_test_raw <= 118349
pred_m, y_m = pred[mask], y_test_raw[mask]
diag = {"n_masked": int((~mask).sum()), "RMSE_masked": round(float(np.sqrt(mean_squared_error(y_m, pred_m))), 1),
        "R2_masked": round(float(r2_score(y_m, pred_m)), 4),
        "RMSE_full": metrics["RMSE"], "R2_full": metrics["R2"]}
print(json.dumps({"outlier_diagnostic_top9_masked": diag}, indent=2))
with open("weights_v4/regression_pytorch_outlier_diag.json", "w") as f: json.dump(diag, f, indent=2)

# per-feature shape functions (interpretability): sweep each feature -2..2 std, others at 0
fresh.eval()
with torch.no_grad():
    grid = np.linspace(-2, 2, 25, dtype=np.float32)
    shapes = {}
    for j, name in enumerate(REG_FEATURES):
        outs = []
        for g in grid:
            xb = torch.zeros(1, len(REG_FEATURES)); xb[0, j] = float(g)
            outs.append(float(fresh.subs[j](xb[:, j:j+1]).item()))
        shapes[name] = [round(v, 4) for v in outs]
    with open("weights_v4/regression_pytorch_shapes.json", "w") as f:
        json.dump({"grid_std": [round(float(g), 3) for g in grid], "shapes": shapes}, f, indent=2)
    print("shape functions saved for", len(shapes), "features")


cleaned: (2213, 28) | Income max: 666,666 | median: 51,373
features: 13 dims (all numeric)
train (1770, 13) test (443, 13) | y_log scaler: mean 10.7459 scale 0.5137
NAM params: 2510


Huber delta=0.5 | epochs 100 | cosine | wd 1e-4 | log1p target, outlier kept
epoch 001 | train-huber 0.8905 | val-RMSE $251,518 | R2 -135.9778 | lr 1.00e-03 | bestRMSE $251,518


epoch 010 | train-huber 0.0979 | val-RMSE $12,638 | R2 0.6542 | lr 9.76e-04 | bestRMSE $12,598


epoch 020 | train-huber 0.0932 | val-RMSE $11,934 | R2 0.6916 | lr 9.05e-04 | bestRMSE $11,830


epoch 030 | train-huber 0.0925 | val-RMSE $11,842 | R2 0.6964 | lr 7.94e-04 | bestRMSE $11,735


...

epoch 060 | train-huber 0.0894 | val-RMSE $11,693 | R2 0.7040 | lr 3.45e-04 | bestRMSE $11,355


epoch 070 | train-huber 0.0875 | val-RMSE $11,573 | R2 0.7100 | lr 2.06e-04 | bestRMSE $11,355


epoch 080 | train-huber 0.0863 | val-RMSE $11,533 | R2 0.7120 | lr 9.55e-05 | bestRMSE $11,307


epoch 090 | train-huber 0.0858 | val-RMSE $11,494 | R2 0.7139 | lr 2.45e-05 | bestRMSE $11,307


epoch 100 | train-huber 0.0856 | val-RMSE $11,490 | R2 0.7141 | lr 0.00e+00 | bestRMSE $11,307
best val-RMSE checkpoint saved ($11,307)
{
  "test_MSE": 127856870.8,
  "RMSE": 11307.4,
  "MAE": 7031.9,
  "R2": 0.7232,
  "n_test": 443,
  "seed": 42,
  "delta": 0.5,
  "target": "log1p+Huber"
}
{
  "outlier_diagnostic_top9_masked": {
    "n_masked": 2,
    "RMSE_masked": 8518.1,
    "R2_masked": 0.8238,
    "RMSE_full": 11307.4,
    "R2_full": 0.7232
  }
}
shape functions saved for 13 features


## Architecture graph (ONNX — source of the REPORT.md image)
One call to the shared `onnx_viz.save_graph` helper exports the trained oracle above to `media/arch_reg_v4.onnx`
(open it in Netron to inspect) and renders `media/arch_reg_v4.png` (input → … → output).

In [2]:
from onnx_viz import save_graph
save_graph(fresh, "arch_reg_v4", 13)


wrote media/arch_reg_v4.onnx + media/arch_reg_v4.png


'media/arch_reg_v4.onnx'